# PubMed Evidence Assistant - Notebook 2: Statistics, Calibration, Error Analysis

Notebook 1 showed that **retrieval is not the bottleneck**: dense search finds the right abstract ~99% of the time, and even with the gold abstract handed over (`oracle`) accuracy was only 0.62 against a 0.55 majority-class baseline. This notebook works on what the numbers actually point at.

**Part A (no GPU, no LLM calls)** - how much of Notebook 1's table can you actually trust? Bootstrap 95% confidence intervals, paired comparisons, citation sanity checks.

**Part B** - a **calibrated decision head**. Instead of letting the LLM *write* "yes/no/maybe" (it has a strong prior, e.g. saying "maybe" to almost everything with no evidence), we read its **probabilities** for the three answers from one forward pass, then fit a small logistic regression on held-out *train* questions to correct the bias. This is a real ML component (features -> classifier -> evaluation with CIs) on top of the LLM, and it is cheaper than generation.

**Part C** - read the errors.

**Run order:** Part A needs only the config cell (run it right after the install). Part B reloads data, the retrievers (embeddings are cached on Drive, so this is fast) and the LLM, then scores questions. It is resumable.
Same Colab setup as Notebook 1: T4 GPU, no API key.

In [ ]:
!pip -q install datasets sentence-transformers rank_bm25 faiss-cpu langchain-core langchain-text-splitters scikit-learn accelerate bitsandbytes

In [ ]:
import os, re, json, time, hashlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path("/content/drive/MyDrive/pubmed_rag") if IN_COLAB else Path("./pubmed_rag")
if IN_COLAB:
    drive.mount("/content/drive")
RESULTS_DIR, CACHE_DIR = ROOT / "results", ROOT / "cache"
for d in (RESULTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- local LLM (no API key, no login) ----------
LLM_ID    = "Qwen/Qwen3-4B-Instruct-2507"   # open, non-thinking, ~8 GB in fp16: fits a free T4
LOAD_4BIT = False                           # True -> 4-bit (~3 GB, slightly lower quality). Use it if you run out of GPU memory
WITH_REASON = False                         # True adds a one-sentence reason (nicer demo, slower evals)
MAX_NEW = 120 if WITH_REASON else 40        # decision + sources is only ~20 tokens
GPU_USD_PER_HOUR = 0.0                      # Colab is free. Enter a cloud GPU's hourly price to estimate cost per query

# ---------- retrieval ----------
EMBED_MODEL  = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge-en-v1.5 query instruction (not used for passages)
N_DISTRACTORS = 4000      # extra abstracts that are NOT the answer. Raise to 20000 for a harder, more realistic search.
K = 5                     # top-k chunks sent to the LLM / scored for retrieval

# ---------- evaluation splits ----------
N_DEV, N_TEST = 150, 150  # stratified by label. Tune on DEV. Touch TEST once, at the end.
SEED = 42

Mounted at /content/drive


## Part A - How much can you trust Notebook 1's table?

With 150 questions, one answer is worth 0.67 accuracy points and the 95% interval on an accuracy near 0.55 is about +/-8 points. This cell reads the saved `gen_dev_*.jsonl` files from Notebook 1 and adds:
- bootstrap 95% CIs for accuracy and macro-F1
- **paired** bootstrap differences between setups (same questions, so tighter than comparing two independent numbers)
- the majority-class baseline **on macro-F1 too** (accuracy flatters the majority class when labels are imbalanced)
- citation sanity: how many doc IDs the model cites per answer, and how often it cites an ID it was never shown

In [ ]:
LABELS = ["yes", "no", "maybe"]
rng = np.random.default_rng(SEED)

def _acc(Y, P):
    return (Y == P).mean(-1)

def _f1_macro(Y, P):
    """Macro-F1 over LABELS along the last axis (vectorised so bootstrapping is fast). Same as sklearn with zero_division=0."""
    f1s = []
    for c in LABELS:
        tp = ((Y == c) & (P == c)).sum(-1)
        fp = ((Y != c) & (P == c)).sum(-1)
        fn = ((Y == c) & (P != c)).sum(-1)
        denom = 2 * tp + fp + fn
        f1s.append(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0))
    return np.mean(f1s, axis=0)

def boot_ci(y, p, fn, n_boot=2000):
    idx = rng.integers(0, len(y), size=(n_boot, len(y)))
    return np.percentile(fn(y[idx], p[idx]), [2.5, 97.5])

def paired_diff(y, p1, p2, fn, n_boot=2000):
    """(point difference, lo, hi) of fn(p1) - fn(p2) on the SAME questions."""
    idx = rng.integers(0, len(y), size=(n_boot, len(y)))
    d = fn(y[idx], p1[idx]) - fn(y[idx], p2[idx])
    return float(fn(y, p1) - fn(y, p2)), *np.percentile(d, [2.5, 97.5])

def load_runs(split="dev"):
    newest = {}
    for p in sorted(RESULTS_DIR.glob(f"gen_{split}_*.jsonl"), key=lambda q: q.stat().st_mtime):
        label = re.sub(r"_[0-9a-f]{6}$", "", p.stem[len("gen_"):])
        newest[label] = p                                   # newest file wins for a given label
    runs = {}
    for label, p in newest.items():
        rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
        runs[label] = pd.DataFrame(rows).set_index("pubid")
    return runs

runs = load_runs("dev")
assert runs, f"No gen_dev_*.jsonl files in {RESULTS_DIR}. Run Notebook 1 first."
common = sorted(set.intersection(*[set(r.index) for r in runs.values()]))
gold = next(iter(runs.values())).loc[common, "gold"].to_numpy()
print(f"{len(runs)} runs, {len(common)} questions in common:", list(runs))

rows = []
for label, r in runs.items():
    r = r.loc[common]
    p = r["decision"].to_numpy()
    a_lo, a_hi = boot_ci(gold, p, _acc)
    f_lo, f_hi = boot_ci(gold, p, _f1_macro)
    is_closed = "closed" in label
    n_cited = r["cited"].apply(len)
    invalid = [not set(c).issubset(set(rv)) for c, rv in zip(r["cited"], r["retrieved"])]
    rows.append({"run": label, "n": len(r),
                 "accuracy": _acc(gold, p), "acc_95ci": f"[{a_lo:.2f}, {a_hi:.2f}]",
                 "macro_f1": _f1_macro(gold, p), "f1_95ci": f"[{f_lo:.2f}, {f_hi:.2f}]",
                 "maybe_recall": float((p[gold == "maybe"] == "maybe").mean()),
                 "avg_ids_cited": np.nan if is_closed else n_cited.mean(),
                 "cites_unseen_id": np.nan if is_closed else float(np.mean(invalid))})
maj = np.array(["yes"] * len(gold))
a_lo, a_hi = boot_ci(gold, maj, _acc); f_lo, f_hi = boot_ci(gold, maj, _f1_macro)
rows.append({"run": "majority_class_yes", "n": len(gold), "accuracy": _acc(gold, maj), "acc_95ci": f"[{a_lo:.2f}, {a_hi:.2f}]",
             "macro_f1": _f1_macro(gold, maj), "f1_95ci": f"[{f_lo:.2f}, {f_hi:.2f}]"})
display(pd.DataFrame(rows).round(3))

def pick(sub):
    return next((k for k in runs if sub in k), None)

print("\nPaired differences on the same questions (95% CI; if it contains 0, you cannot claim a difference):")
pairs = [("baseline", "closed"), ("best", "baseline"), ("oracle", "best"), ("oracle", "closed")]
out = []
for a, b in pairs:
    ka, kb = pick(a), pick(b)
    if ka and kb:
        pa_, pb_ = runs[ka].loc[common, "decision"].to_numpy(), runs[kb].loc[common, "decision"].to_numpy()
        for name, fn in (("accuracy", _acc), ("macro_f1", _f1_macro)):
            d, lo, hi = paired_diff(gold, pa_, pb_, fn)
            out.append({"comparison": f"{ka} - {kb}", "metric": name, "diff": d, "ci_low": lo, "ci_high": hi, "excludes_0": lo > 0 or hi < 0})
display(pd.DataFrame(out).round(3))

for label, r in runs.items():
    print(f"\n{label}  (rows = gold, columns = predicted)")
    print(pd.crosstab(pd.Series(gold, name="gold"), pd.Series(r.loc[common, "decision"].to_numpy(), name="pred")).reindex(index=LABELS, columns=LABELS + ["invalid"], fill_value=0))

4 runs, 150 questions in common: ['dev_closed_book', 'dev_baseline_rcts1000_dense', 'dev_best_whole_abstract_dense', 'dev_oracle']


,run,n,accuracy,acc_95ci,macro_f1,f1_95ci,maybe_recall,avg_ids_cited,cites_unseen_id
0,dev_closed_book,150,0.173,"[0.11, 0.24]",0.149,"[0.10, 0.21]",0.812,NaN,NaN
1,dev_baseline_rcts1000_dense,150,0.527,"[0.45, 0.61]",0.411,"[0.34, 0.48]",0.375,1.547,0.040
2,dev_best_whole_abstract_dense,150,0.587,"[0.51, 0.67]",0.477,"[0.40, 0.56]",0.500,1.733,0.107
3,dev_oracle,150,0.620,"[0.54, 0.70]",0.535,"[0.46, 0.61]",0.438,1.000,0.000
4,majority_class_yes,150,0.553,"[0.48, 0.63]",0.237,"[0.21, 0.26]",NaN,NaN,NaN



Paired differences on the same questions (95% CI; if it contains 0, you cannot claim a difference):


,comparison,metric,diff,ci_low,ci_high,excludes_0
0,dev_baseline_rcts1000_dense - dev_closed_book,accuracy,0.353,0.260,0.447,True
1,dev_baseline_rcts1000_dense - dev_closed_book,macro_f1,0.261,0.179,0.344,True
2,dev_best_whole_abstract_dense - dev_baseline_r...,accuracy,0.060,0.007,0.113,True
3,dev_best_whole_abstract_dense - dev_baseline_r...,macro_f1,0.066,0.006,0.130,True
4,dev_oracle - dev_best_whole_abstract_dense,accuracy,0.033,-0.033,0.100,False
5,dev_oracle - dev_best_whole_abstract_dense,macro_f1,0.058,-0.009,0.124,False
6,dev_oracle - dev_closed_book,accuracy,0.447,0.347,0.533,True
7,dev_oracle - dev_closed_book,macro_f1,0.386,0.298,0.465,True



dev_closed_book  (rows = gold, columns = predicted)
pred   yes  no  maybe  invalid
gold                          
yes     12   2     69        0
no      10   1     40        0
maybe    3   0     13        0

dev_baseline_rcts1000_dense  (rows = gold, columns = predicted)
pred   yes  no  maybe  invalid
gold                          
yes     63   1     19        0
no      19  10     22        0
maybe    9   1      6        0

dev_best_whole_abstract_dense  (rows = gold, columns = predicted)
pred   yes  no  maybe  invalid
gold                          
yes     67   1     15        0
no      14  13     24        0
maybe    7   1      8        0

dev_oracle  (rows = gold, columns = predicted)
pred   yes  no  maybe  invalid
gold                          
yes     63   3     17        0
no       5  23     23        0
maybe    8   1      7        0


## Part B - Calibrated decision head

First, reload the pieces from Notebook 1 (same cells, so the splits and prompts are identical). Embeddings come from the Drive cache; only the BM25 indexes are rebuilt.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def to_abstract(ctx):
    """PubMedQA 'context' = parallel lists: contexts (text), labels (section names), meshes."""
    parts = []
    for label, text in zip(ctx["labels"], ctx["contexts"]):
        parts.append(f"{str(label).title()}: {text}" if label else text)
    return "\n".join(parts)

labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
questions = pd.DataFrame([{
    "pubid": str(r["pubid"]), "question": r["question"], "decision": r["final_decision"],
    "abstract": to_abstract(r["context"]),
    "long_answer": r["long_answer"],            # error analysis ONLY - never send to the LLM
} for r in labeled])

pool = dict(zip(questions.pubid, questions.abstract))
extra = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train", streaming=True).take(N_DISTRACTORS)
for r in extra:
    pool.setdefault(str(r["pubid"]), to_abstract(r["context"]))
docs = [{"doc_id": k, "text": v} for k, v in pool.items()]

# fixed, saved splits so results are reproducible
dev_idx, rest_idx = train_test_split(questions.index, train_size=N_DEV, stratify=questions.decision, random_state=SEED)
test_idx, _ = train_test_split(rest_idx, train_size=N_TEST, stratify=questions.loc[rest_idx, "decision"], random_state=SEED)
dev_df, test_df = questions.loc[dev_idx], questions.loc[test_idx]
pd.concat([dev_df.assign(split="dev"), test_df.assign(split="test")])[["pubid", "split"]].to_csv(RESULTS_DIR / "splits.csv", index=False)

print(f"{len(questions)} labelled questions | {len(docs)} abstracts in the pool | avg {np.mean([len(d['text']) for d in docs]):.0f} chars")
print("label share:", questions.decision.value_counts(normalize=True).round(3).to_dict())
print(f"majority-class (always 'yes') accuracy on all labelled: {(questions.decision == 'yes').mean():.3f}  <- the number to beat")

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

1000 labelled questions | 5000 abstracts in the pool | avg 1385 chars
label share: {'yes': 0.552, 'no': 0.338, 'maybe': 0.11}
majority-class (always 'yes') accuracy on all labelled: 0.552  <- the number to beat


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_whole(docs):
    return [{"doc_id": d["doc_id"], "text": d["text"]} for d in docs]

def make_rcts(chunk_chars, overlap):
    def _chunk(docs):
        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_chars, chunk_overlap=overlap)
        lc_docs = [Document(page_content=d["text"], metadata={"doc_id": d["doc_id"]}) for d in docs]
        return [{"doc_id": c.metadata["doc_id"], "text": c.page_content} for c in splitter.split_documents(lc_docs)]
    return _chunk

CHUNKERS = {"whole_abstract": chunk_whole, "rcts_500": make_rcts(500, 100), "rcts_1000": make_rcts(1000, 200)}
for name, fn in CHUNKERS.items():
    print(f"{name:15s} -> {len(fn(docs[:500]))} chunks from 500 docs")

whole_abstract  -> 500 chunks from 500 docs
rcts_500        -> 2046 chunks from 500 docs
rcts_1000       -> 1035 chunks from 500 docs


In [ ]:
import faiss, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 512
if torch.cuda.is_available():
    embedder.half()                                  # fp16 on the T4: ~2x faster, same quality for retrieval

def tok(s):
    return re.findall(r"\w+", s.lower())

def embed_cached(texts, tag):
    key = hashlib.md5((EMBED_MODEL + "||" + "\x1f".join(texts)).encode("utf-8")).hexdigest()[:16]
    path = CACHE_DIR / f"emb_{tag}_{key}.npy"
    if path.exists():
        return np.load(path)
    emb = np.asarray(embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True), dtype="float32")
    np.save(path, emb)
    return emb

# query embeddings, computed once and reused by every retriever / mode
Q_EMB = np.asarray(embedder.encode([QUERY_PREFIX + q for q in questions.question], normalize_embeddings=True,
                                   batch_size=64, show_progress_bar=False), dtype="float32")
QIDX = {pid: i for i, pid in enumerate(questions.pubid)}

class Retriever:
    def __init__(self, chunks, tag):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        self.bm25 = BM25Okapi([tok(t) for t in texts])
        emb = embed_cached(texts, tag)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def _dense(self, qvec, n):
        _, ids = self.index.search(np.asarray(qvec, dtype="float32").reshape(1, -1), n)
        return [int(i) for i in ids[0] if i >= 0]

    def _bm25(self, q, n):
        scores = self.bm25.get_scores(tok(q))
        return [int(i) for i in np.argsort(scores)[::-1][:n] if scores[i] > 0]

    def search(self, q, qvec, k=K, mode="hybrid", pool=30):
        if mode == "dense":
            ids = self._dense(qvec, k)
        elif mode == "bm25":
            ids = self._bm25(q, k)
        else:                                         # hybrid = reciprocal rank fusion of both rankings
            fused = defaultdict(float)
            for ranking in (self._dense(qvec, pool), self._bm25(q, pool)):
                for rank, i in enumerate(ranking):
                    fused[i] += 1.0 / (60 + rank + 1)
            ids = sorted(fused, key=fused.get, reverse=True)[:k]
        return [self.chunks[i] for i in ids]

RETRIEVERS = {}
for name, fn in {k: CHUNKERS[k] for k in ("whole_abstract", "rcts_1000")}.items():
    chunks = fn(docs)
    RETRIEVERS[name] = Retriever(chunks, name)
    print(f"{name}: {len(chunks)} chunks indexed")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

whole_abstract: 5000 chunks indexed
rcts_1000: 10533 chunks indexed


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)

def load_llm():
    kw = {"device_map": "auto"}
    if LOAD_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    last = None
    for dtype_arg in ("dtype", "torch_dtype"):        # this argument was renamed across transformers versions
        try:
            return AutoModelForCausalLM.from_pretrained(LLM_ID, **{dtype_arg: torch.float16}, **kw).eval()
        except TypeError as e:
            last = e
    raise last

llm = load_llm()

@torch.inference_mode()
def llm_chat(system, user):
    """Local generation, greedy. Returns (text, prompt_tokens, completion_tokens)."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    n_in = enc["input_ids"].shape[1]
    gen = out[0][n_in:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip(), int(n_in), int(gen.shape[0])

# sanity check
t0 = time.perf_counter()
print(repr(llm_chat("You are a helpful assistant.", "Reply with exactly: Decision: yes")[0]), f"| {time.perf_counter() - t0:.1f}s")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

'Decision: yes' | 0.7s


In [ ]:
FORMAT = "Decision: <yes|no|maybe>\nSources: <doc_id>, <doc_id>" + ("\nReason: <one sentence>" if WITH_REASON else "")
SYSTEM_RAG = (
    "You are a biomedical literature assistant. Answer the research question using ONLY the provided abstracts.\n"
    "Decide: yes, no, or maybe. Use 'maybe' only when the abstracts report mixed or inconclusive evidence.\n"
    "In Sources list the doc_id of every abstract you relied on.\n"
    "Reply in exactly this format and nothing else:\n" + FORMAT + "\n"
    "This tool summarises published research. It does not give medical advice."
)
SYSTEM_CLOSED = (
    "You are a biomedical literature assistant. Answer the research question from your general biomedical knowledge.\n"
    "Decide: yes, no, or maybe. Use 'maybe' only when the evidence is mixed or inconclusive.\n"
    "Reply in exactly this format and nothing else (write 'none' for Sources):\n" + FORMAT + "\n"
    "This tool summarises published research. It does not give medical advice."
)

def format_context(hits):
    by_doc = {}
    for h in hits:
        by_doc.setdefault(h["doc_id"], []).append(h["text"])
    return "\n\n".join(f"[doc_id: {d}]\n" + "\n...\n".join(ts) for d, ts in by_doc.items())

def parse_answer(raw):
    m = re.search(r"decision\s*:\s*\**\s*(yes|no|maybe)", raw, re.I)
    if m:
        s = re.search(r"sources?\s*:\s*(.*)", raw, re.I)
        r = re.search(r"reason\s*:\s*(.*)", raw, re.I | re.S)
        return m.group(1).lower(), (re.findall(r"\d+", s.group(1)) if s else []), (r.group(1).strip() if r else ""), True
    m = re.search(r"\b(yes|no|maybe)\b", raw.lower())      # fallback: counts as a parse failure
    return (m.group(1) if m else "invalid"), [], raw[:200], False

def answer(row, setup, retriever=None, search_mode="hybrid", k=K):
    t0 = time.perf_counter()
    retrieved, gold_retrieved = [], None
    if setup == "closed_book":
        system, user = SYSTEM_CLOSED, f"QUESTION: {row.question}"
    elif setup == "oracle":
        system, user = SYSTEM_RAG, f"ABSTRACTS:\n[doc_id: {row.pubid}]\n{row.abstract}\n\nQUESTION: {row.question}"
        retrieved = [row.pubid]
    else:
        hits = retriever.search(row.question, Q_EMB[QIDX[row.pubid]], k=k, mode=search_mode)
        retrieved = list(dict.fromkeys(h["doc_id"] for h in hits))
        gold_retrieved = row.pubid in retrieved
        system, user = SYSTEM_RAG, f"ABSTRACTS:\n{format_context(hits)}\n\nQUESTION: {row.question}"
    raw, tin, tout = llm_chat(system, user)
    decision, cited, why, ok = parse_answer(raw)
    latency = time.perf_counter() - t0
    return {"decision": decision, "rationale": why, "cited": cited, "parsed_ok": ok,
            "retrieved": retrieved, "gold_retrieved": gold_retrieved,
            "cited_gold": (row.pubid in cited) if setup != "closed_book" else None,
            "latency_s": latency, "tokens_in": tin, "tokens_out": tout,
            "cost_usd": latency / 3600 * GPU_USD_PER_HOUR}

# one live test before running the whole eval
test_row = next(dev_df.head(1).itertuples())
out = answer(test_row, "rag", RETRIEVERS["rcts_1000"], "hybrid")
print("gold:", test_row.decision, "| model:", out["decision"], "| cited:", out["cited"], "| gold retrieved:", out["gold_retrieved"])
print(f"{out['latency_s']:.1f}s | {out['tokens_in']}+{out['tokens_out']} tokens")

gold: yes | model: yes | cited: ['21123461'] | gold retrieved: True
1.4s | 1009+16 tokens


### Splits and prompts

`train_df` is a stratified sample of the labelled questions that are **in neither dev nor test**. It is used only to fit the 3-feature calibrator. Dev and test are exactly the ones from Notebook 1.

Four setups are scored: `closed_book`, `rag_chunks` (1000-character chunks, dense, k=5: the cheap option), `rag_whole` (whole abstracts, dense, k=5: ~2x the prompt tokens), and `oracle`.

In [ ]:
N_TRAIN = 300                 # questions used to fit the calibrator
EVAL_SPLIT = "test"            # switch to "test" ONCE, at the very end

rest = questions.index.difference(dev_df.index).difference(test_df.index)
train_idx, _ = train_test_split(rest, train_size=N_TRAIN, stratify=questions.loc[rest, "decision"], random_state=SEED)
train_df = questions.loc[train_idx]
SPLITS = {"train": train_df, "dev": dev_df, "test": test_df}
print({k: len(v) for k, v in SPLITS.items()}, "| train label share:", train_df.decision.value_counts(normalize=True).round(2).to_dict())

RAG_SETUPS = {"rag_chunks": ("rcts_1000", "dense"), "rag_whole": ("whole_abstract", "dense")}
SETUPS = ["closed_book", "rag_chunks", "rag_whole", "oracle"]

def make_prompt(row, setup):
    """Same prompts as Notebook 1's answer(). Returns (system, user, gold_retrieved)."""
    if setup == "closed_book":
        return SYSTEM_CLOSED, f"QUESTION: {row.question}", None
    if setup == "oracle":
        return SYSTEM_RAG, f"ABSTRACTS:\n[doc_id: {row.pubid}]\n{row.abstract}\n\nQUESTION: {row.question}", True
    chunker, mode = RAG_SETUPS[setup]
    hits = RETRIEVERS[chunker].search(row.question, Q_EMB[QIDX[row.pubid]], k=K, mode=mode)
    return SYSTEM_RAG, f"ABSTRACTS:\n{format_context(hits)}\n\nQUESTION: {row.question}", row.pubid in {h["doc_id"] for h in hits}

{'train': 300, 'dev': 150, 'test': 150} | train label share: {'yes': 0.55, 'no': 0.34, 'maybe': 0.11}


### Reading the model's probabilities

We end the prompt with `Decision:` and read the next-token distribution over *yes / no / maybe* (summing the few spelling variants such as ` yes`, `Yes`). One forward pass, no text generation, so it is faster than Notebook 1's runs and can never fail to parse.

In [ ]:
from scipy.special import log_softmax, logsumexp

CLASSES = ["yes", "no", "maybe"]

def _class_token_ids(word):
    ids = set()
    for v in (word, " " + word, word.capitalize(), " " + word.capitalize()):
        enc = tokenizer.encode(v, add_special_tokens=False)
        if enc:
            ids.add(int(enc[0]))                  # first token of each spelling variant
    return sorted(ids)

CLASS_IDS = [_class_token_ids(c) for c in CLASSES]
flat = [i for ids in CLASS_IDS for i in ids]
assert len(flat) == len(set(flat)), f"class tokens overlap: {CLASS_IDS}"
print({c: tokenizer.convert_ids_to_tokens(ids) for c, ids in zip(CLASSES, CLASS_IDS)})

@torch.inference_mode()
def score_decision(system, user):
    """One forward pass. Returns (log-probs of yes/no/maybe as the next token after 'Decision:', prompt_tokens)."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True) + "Decision:"
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(llm.device)
    try:
        out = llm(**enc, logits_to_keep=1)        # only compute the last position's logits (saves memory)
    except TypeError:
        out = llm(**enc)                          # older transformers
    logp = log_softmax(out.logits[0, -1].float().cpu().numpy())
    return [float(logsumexp(logp[ids])) for ids in CLASS_IDS], int(enc["input_ids"].shape[1])

# sanity check: a clear factual question should put most mass on 'yes'
lp, _ = score_decision(SYSTEM_CLOSED, "QUESTION: Is aspirin a nonsteroidal anti-inflammatory drug?")
print("P(yes/no/maybe) =", np.round(np.exp(lp - logsumexp(lp)), 3))

{'yes': ['ĠYes', 'Yes', 'yes', 'Ġyes'], 'no': ['Ġno', 'no', 'ĠNo', 'No'], 'maybe': ['Ġmaybe', 'ĠMaybe', 'Maybe', 'maybe']}
P(yes/no/maybe) = [1. 0. 0.]


### Score train + eval split for every setup

Each score is appended to a JSONL on Drive, so a Colab disconnect just resumes. This is the long step (on the order of half an hour in total, mostly the two RAG setups); `N_TRAIN` is the main lever if you need it faster.

In [ ]:
def score_hash():
    return hashlib.md5((SYSTEM_RAG + SYSTEM_CLOSED + LLM_ID + str(K) + str(LOAD_4BIT)).encode()).hexdigest()[:6]

def score_setup(split, setup):
    df = SPLITS[split]
    path = RESULTS_DIR / f"score_{split}_{setup}_{score_hash()}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            rec = json.loads(line)
            done[rec["pubid"]] = rec
    rows = []
    with path.open("a", encoding="utf-8") as f:
        for row in tqdm(list(df.itertuples()), desc=f"{split}/{setup}"):
            if row.pubid in done:
                rows.append(done[row.pubid])
                continue
            system, user, gold_retrieved = make_prompt(row, setup)
            lp, n_in = score_decision(system, user)
            rec = {"pubid": row.pubid, "gold": row.decision, "lp": lp, "tokens_in": n_in, "gold_retrieved": gold_retrieved}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            rows.append(rec)
    return pd.DataFrame(rows).set_index("pubid")

FEATS = {}
for split in ("train", EVAL_SPLIT):
    for setup in SETUPS:
        FEATS[(split, setup)] = score_setup(split, setup)
print({k: len(v) for k, v in FEATS.items()})

train/closed_book:   0%|          | 0/300 [00:00<?, ?it/s]

train/rag_chunks:   0%|          | 0/300 [00:00<?, ?it/s]

train/rag_whole:   0%|          | 0/300 [00:00<?, ?it/s]

train/oracle:   0%|          | 0/300 [00:00<?, ?it/s]

test/closed_book:   0%|          | 0/150 [00:00<?, ?it/s]

test/rag_chunks:   0%|          | 0/150 [00:00<?, ?it/s]

test/rag_whole:   0%|          | 0/150 [00:00<?, ?it/s]

test/oracle:   0%|          | 0/150 [00:00<?, ?it/s]

{('train', 'closed_book'): 300, ('train', 'rag_chunks'): 300, ('train', 'rag_whole'): 300, ('train', 'oracle'): 300, ('test', 'closed_book'): 150, ('test', 'rag_chunks'): 150, ('test', 'rag_whole'): 150, ('test', 'oracle'): 150}


### Calibrate and evaluate

For each setup: **argmax** = take the model's most probable answer as-is; **calibrated** = a multinomial logistic regression (`class_weight="balanced"`) fitted on the `train` features, applied to the eval split. Two calibrators are fitted: `calibrated` (balanced class weights: trades some accuracy for recall on the rare "no" and "maybe" classes, so watch macro-F1) and `calibrated_unweighted` (optimises plain accuracy). If the balanced one predicts "maybe" for nearly everything, the probabilities carry little signal; say so in your README rather than hiding it. All intervals are bootstrap 95% CIs.

In [ ]:
from sklearn.linear_model import LogisticRegression

def norm_lp(df):
    lp = np.array(df["lp"].tolist(), dtype=float)
    return lp - logsumexp(lp, axis=1, keepdims=True)            # renormalise over the three classes

preds, y_eval = {}, None
for setup in SETUPS:
    tr, ev = FEATS[("train", setup)], FEATS[(EVAL_SPLIT, setup)]
    Xtr, ytr, Xev, yev = norm_lp(tr), tr["gold"].to_numpy(), norm_lp(ev), ev["gold"].to_numpy()
    assert y_eval is None or (yev == y_eval).all(), "eval questions differ between setups"
    y_eval = yev
    clf_bal = LogisticRegression(class_weight="balanced", C=1.0, max_iter=2000).fit(Xtr, ytr)
    clf_raw = LogisticRegression(class_weight=None, C=1.0, max_iter=2000).fit(Xtr, ytr)
    preds[(setup, "argmax")] = np.array(CLASSES)[Xev.argmax(1)]
    preds[(setup, "calibrated")] = clf_bal.predict(Xev)                 # optimises for macro-F1 (rare classes matter)
    preds[(setup, "calibrated_unweighted")] = clf_raw.predict(Xev)      # optimises for plain accuracy

rows = []
for (setup, method), p in preds.items():
    a_lo, a_hi = boot_ci(y_eval, p, _acc); f_lo, f_hi = boot_ci(y_eval, p, _f1_macro)
    rows.append({"setup": setup, "method": method, "n": len(p),
                 "accuracy": _acc(y_eval, p), "acc_95ci": f"[{a_lo:.2f}, {a_hi:.2f}]",
                 "macro_f1": _f1_macro(y_eval, p), "f1_95ci": f"[{f_lo:.2f}, {f_hi:.2f}]",
                 "maybe_recall": float((p[y_eval == "maybe"] == "maybe").mean()),
                 "avg_prompt_tokens": FEATS[(EVAL_SPLIT, setup)]["tokens_in"].mean()})
maj = np.array(["yes"] * len(y_eval))
rows.append({"setup": "majority_class_yes", "method": "-", "n": len(y_eval), "accuracy": _acc(y_eval, maj), "macro_f1": _f1_macro(y_eval, maj)})
results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / f"calibration_{EVAL_SPLIT}.csv", index=False)
display(results.round(3))

print("\nPaired differences (same questions). If the 95% CI contains 0 you cannot claim a difference:")
cmp_rows = []
def add(name, p1, p2):
    for metric, fn in (("accuracy", _acc), ("macro_f1", _f1_macro)):
        d, lo, hi = paired_diff(y_eval, p1, p2, fn)
        cmp_rows.append({"comparison": name, "metric": metric, "diff": d, "ci_low": lo, "ci_high": hi, "excludes_0": lo > 0 or hi < 0})
for s in SETUPS:
    add(f"{s}: calibrated - argmax", preds[(s, "calibrated")], preds[(s, "argmax")])
    add(f"{s}: calibrated_unweighted - argmax", preds[(s, "calibrated_unweighted")], preds[(s, "argmax")])
add("rag_whole - rag_chunks (calibrated)", preds[("rag_whole", "calibrated")], preds[("rag_chunks", "calibrated")])
add("rag_chunks - closed_book (calibrated)", preds[("rag_chunks", "calibrated")], preds[("closed_book", "calibrated")])
add("oracle - rag_whole (calibrated)", preds[("oracle", "calibrated")], preds[("rag_whole", "calibrated")])
display(pd.DataFrame(cmp_rows).round(3))

,setup,method,n,accuracy,acc_95ci,macro_f1,f1_95ci,maybe_recall,avg_prompt_tokens
0,closed_book,argmax,150,0.207,"[0.15, 0.27]",0.187,"[0.13, 0.25]",0.875,126.073
1,closed_book,calibrated,150,0.480,"[0.40, 0.56]",0.367,"[0.30, 0.43]",0.062,126.073
2,closed_book,calibrated_unweighted,150,0.547,"[0.47, 0.63]",0.285,"[0.23, 0.34]",0.000,126.073
3,rag_chunks,argmax,150,0.500,"[0.42, 0.57]",0.366,"[0.30, 0.44]",0.500,998.553
4,rag_chunks,calibrated,150,0.553,"[0.47, 0.63]",0.496,"[0.42, 0.57]",0.438,998.553
5,rag_chunks,calibrated_unweighted,150,0.647,"[0.57, 0.73]",0.441,"[0.39, 0.49]",0.000,998.553
6,rag_whole,argmax,150,0.527,"[0.45, 0.61]",0.413,"[0.34, 0.49]",0.500,1879.820
7,rag_whole,calibrated,150,0.607,"[0.53, 0.68]",0.534,"[0.46, 0.60]",0.375,1879.820
8,rag_whole,calibrated_unweighted,150,0.693,"[0.62, 0.77]",0.482,"[0.43, 0.53]",0.000,1879.820
9,oracle,argmax,150,0.587,"[0.51, 0.67]",0.484,"[0.41, 0.56]",0.312,481.100



Paired differences (same questions). If the 95% CI contains 0 you cannot claim a difference:


,comparison,metric,diff,ci_low,ci_high,excludes_0
0,closed_book: calibrated - argmax,accuracy,0.273,0.167,0.380,True
1,closed_book: calibrated - argmax,macro_f1,0.180,0.086,0.277,True
2,closed_book: calibrated_unweighted - argmax,accuracy,0.340,0.233,0.447,True
3,closed_book: calibrated_unweighted - argmax,macro_f1,0.098,0.028,0.168,True
4,rag_chunks: calibrated - argmax,accuracy,0.053,-0.033,0.147,False
5,rag_chunks: calibrated - argmax,macro_f1,0.130,0.041,0.218,True
6,rag_chunks: calibrated_unweighted - argmax,accuracy,0.147,0.067,0.227,True
7,rag_chunks: calibrated_unweighted - argmax,macro_f1,0.075,0.004,0.148,True
8,rag_whole: calibrated - argmax,accuracy,0.080,-0.007,0.160,False
9,rag_whole: calibrated - argmax,macro_f1,0.120,0.039,0.205,True


## Part C - Read the errors

Numbers tell you *how often* it fails; reading tells you *why*. Look at ~12 wrong calibrated answers for `rag_whole`, next to the paper's own conclusion (which the model never saw). Sort each into: retrieval miss, model ignored the evidence, ambiguous/noisy label, or genuine "maybe" confusion, and write the tally in your README.

In [ ]:
setup = "rag_whole"
ev = FEATS[(EVAL_SPLIT, setup)].copy()
ev["pred"] = preds[(setup, "calibrated")]
qi = questions.set_index("pubid")
ev["question"], ev["conclusion"] = qi.loc[ev.index, "question"], qi.loc[ev.index, "long_answer"].str[:280]
wrong = ev[ev["gold"] != ev["pred"]]
print(f"{len(wrong)} wrong of {len(ev)} | gold retrieved in {int(wrong['gold_retrieved'].sum())} of the wrong ones\n")
print(pd.crosstab(ev["gold"], ev["pred"]).reindex(index=LABELS, columns=LABELS, fill_value=0), "\n")
for pid, r in wrong.head(12).iterrows():
    print(f"[{pid}] gold={r['gold']} pred={r['pred']} retrieved_gold={r['gold_retrieved']}\n  Q: {r['question']}\n  paper's conclusion: {r['conclusion']}\n")

59 wrong of 150 | gold retrieved in 59 of the wrong ones

pred   yes  no  maybe
gold                 
yes     52  11     20
no       2  33     16
maybe    6   4      6 

[12094116] gold=yes pred=maybe retrieved_gold=True
  Q: Is muscle power related to running speed with changes of direction?
  paper's conclusion: The relationships between leg muscle power and change-of-direction speed were not consistent. Reactive strength as measured by the drop jump appears to have some importance for lateral change-of-direction speed, possibly because of similar push-off actions. It was concluded that 

[24073931] gold=no pred=maybe retrieved_gold=True
  Q: Is the covering of the resection margin after distal pancreatectomy advantageous?
  paper's conclusion: The results show no significant difference in the fistula rate after covering of the resection margin after distal pancreatectomy, which contributes to the picture of an unsolved problem.

[23848044] gold=yes pred=maybe retrieved_gold=True
  Q

## What to do next

1. Read Part A first. Anything whose CI includes 0 is **not** a claim you can make. Write only what survives.
2. When you're happy, set `EVAL_SPLIT = "test"` in the split cell, re-run from there **once**, and take the README numbers from that run.
3. **Then stop tuning the model and build the product layer** (this is what the JD's "build products, not prototypes" is really asking for):
   - FastAPI service: `/ask` returns decision, calibrated probabilities and the cited abstracts
   - a Streamlit (or React) demo with a feedback button
   - Dockerfile + GitHub Actions running tests and a small eval as a regression gate
   - request logging for latency/cost, and a short responsible-AI section in the README
4. Optional stretch if time allows: a QLoRA fine-tune of the 4B model on the labelled train split, compared against this calibrated head.